<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>

<h2 align="center">Roteiro de Atividade Prática</h2>

<br>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Projeto Análise Exploratória de Dados  
**Tema da Semana:** Projeto de Análise Exploratória de Dados: Planejamento e Exploração Inicial  
**Semana 6**  
**Aula 3:** Exploração inicial dos dados do projeto

<br>

# Exploração inicial dos dados do projeto: do plano às evidências

**Situação profissional**

Na Aula 1, vocês escolheram a pergunta que orienta o projeto. Na Aula 2, transformaram essa pergunta em um plano de investigação.

> **Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?**

Agora, como **assistentes de dados da equipe comercial**, vocês vão executar a parte do plano prevista para esta aula: caracterizar informações centrais e buscar evidências nas frentes de **Atração** e **Conversão**.

A tarefa não é produzir gráficos por produzir. Cada estatística, visualização e correlação deve ajudar a examinar uma parte da pergunta investigativa.

**Missão da aula**

> **Executar a exploração inicial planejada, examinar os resultados e registrar evidências que serão interpretadas e integradas na Aula 4.**

## Passo a passo

- Preparem os arquivos e recuperem o conjunto de trabalho com as decisões das Aulas 1 e 2.
- Caracterizem movimento, aproveitamento e espera com estatísticas descritivas e distribuições.
- Investiguem a relação entre investimento em divulgação e entradas.
- Investiguem como disponibilidade, espera e horas de equipe se relacionam com o aproveitamento do movimento.
- Examinem, com boxplots, as distribuições de horas de equipe e de aproveitamento entre as lojas.
- Determinem a faixa de entradas aplicando o critério fornecido e confiram a faixa usada nas comparações.
- Examinem a relação entre horas de equipe e aproveitamento em quatro níveis: rede inteira, cada loja, uma faixa comum de movimento e cada loja dentro dessa faixa.
- Registrem como a direção e a intensidade da relação mudam entre os quatro níveis.
- Consolidem as evidências produzidas e identifiquem a frente que permanece pendente para a Aula 4.
- Salvem o notebook preenchido.


## Preparação

Arquivos usados nesta atividade:

- `DADOS3EMC3B1S6A3_aluno.ipynb`
- `rede_papelarias.csv`

1. Salvem os dois arquivos **na mesma pasta**.
2. Abram o arquivo `.ipynb` no **Visual Studio Code**.
3. Se solicitado, selecionem um **kernel Python** disponível.
4. Executem a célula abaixo.

O código recupera as decisões já tomadas no projeto: retira somente a cópia duplicada, preserva os registros em que `espera_media_min` está ausente e recria os indicadores definidos na Aula 2.

> Os registros com `espera_media_min` ausente continuam válidos para análises que não dependem dessa variável. Eles não devem ser tratados como espera igual a zero.

> Se o professor optar pelo Google Colab, façam o upload de `rede_papelarias.csv` para a sessão antes de executar a célula.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

arquivo_csv = Path("rede_papelarias.csv")

if not arquivo_csv.is_file():
    raise FileNotFoundError(
        "Arquivo 'rede_papelarias.csv' não encontrado. "
        "Salve o CSV na mesma pasta do notebook ou faça o upload para a sessão."
    )

dados = pd.read_csv(arquivo_csv)
dados_trabalho = dados.drop_duplicates().copy()

# Conferência dos denominadores usados nos indicadores
entradas_invalidas = (
    dados_trabalho["entradas"].isna() | (dados_trabalho["entradas"] <= 0)
).sum()
receitas_invalidas = (
    dados_trabalho["receita_liquida"].isna() | (dados_trabalho["receita_liquida"] <= 0)
).sum()

if entradas_invalidas or receitas_invalidas:
    raise ValueError(
        "Há denominadores ausentes ou não positivos para o cálculo dos indicadores."
    )

# Indicadores definidos na Aula 2, calculados para cada loja em cada dia
dados_trabalho["cupons_por_100_entradas"] = (
    dados_trabalho["cupons_emitidos"] / dados_trabalho["entradas"] * 100
)
dados_trabalho["margem_bruta_pct"] = (
    dados_trabalho["lucro_bruto"] / dados_trabalho["receita_liquida"] * 100
)

print("Registros no arquivo:", len(dados))
print("Registros únicos no conjunto de trabalho:", len(dados_trabalho))
print("Valores ausentes em espera_media_min:", dados_trabalho["espera_media_min"].isna().sum())


def dispersao_com_r(df, x, y, titulo, rotulo_x, rotulo_y):
    """Exibe a dispersão e calcula Pearson usando somente pares válidos."""
    pares = df[[x, y]].dropna()
    r = pares[x].corr(pares[y])

    print(f"Pares válidos: {len(pares)} | r = {r:+.3f}")
    plt.figure(figsize=(7, 4.5))
    plt.scatter(pares[x], pares[y], alpha=0.35, s=18)
    plt.xlabel(rotulo_x)
    plt.ylabel(rotulo_y)
    plt.title(titulo)
    plt.grid(alpha=0.15)
    plt.tight_layout()
    plt.show()
    return {"n": len(pares), "r": r}

print("\nConjunto de trabalho preparado.")


## Etapa 1 — Caracterizem informações centrais

Antes de interpretar relações entre variáveis, examinem três variáveis e indicadores centrais do plano da Aula 2:

- `entradas`: movimento recebido pela loja;
- `cupons_por_100_entradas`: aproveitamento do movimento em compras;
- `espera_media_min`: condição operacional ligada ao atendimento.

Executem a célula abaixo. Ela apresenta, para cada variável, **número de registros válidos, média, mediana, desvio padrão, mínimo e máximo**, além de um histograma.

> Para `cupons_por_100_entradas`, a média mostrada é a **média dos valores calculados para cada loja-dia**. Ela não representa uma taxa única calculada para toda a rede.

In [ ]:
variaveis_caracterizacao = {
    "entradas": "Entradas de pessoas",
    "cupons_por_100_entradas": "Cupons por 100 entradas",
    "espera_media_min": "Espera média (min)"
}

linhas = []
for coluna, rotulo in variaveis_caracterizacao.items():
    serie = dados_trabalho[coluna].dropna()
    linhas.append({
        "medida": rotulo,
        "n": len(serie),
        "média": serie.mean(),
        "mediana": serie.median(),
        "desvio padrão": serie.std(),
        "mínimo": serie.min(),
        "máximo": serie.max(),
    })

resumo = pd.DataFrame(linhas).set_index("medida").round(2)
display(resumo)

for coluna, rotulo in variaveis_caracterizacao.items():
    serie = dados_trabalho[coluna].dropna()
    plt.figure(figsize=(7, 4.2))
    plt.hist(serie, bins=25, edgecolor="white")
    plt.xlabel(rotulo)
    plt.ylabel("Número de loja-dias")
    plt.title(f"Distribuição — {rotulo}")
    plt.grid(axis="y", alpha=0.15)
    plt.tight_layout()
    plt.show()


### Registro 1 — caracterização

**1. Registrem uma observação útil sobre cada variável, relacionando o histograma às medidas da tabela. No conjunto das três respostas, utilizem pelo menos uma medida de tendência central e uma de dispersão e expliquem o que elas mostram.**

**Entradas de pessoas:**  
**Resposta:**


**Cupons por 100 entradas:**  
**Resposta:**


**Espera média:**  
**Resposta:**


**2. Por que os 24 registros sem `espera_media_min` não devem ser interpretados como espera igual a zero?**

**Resposta:**

## Etapa 2 — Investiguem a frente Atração

Pergunta desta frente:

> **Como o investimento em divulgação se relaciona com o número de entradas nas lojas?**

Executem a célula abaixo e examinem **o gráfico de dispersão e o valor de `r` em conjunto**.

In [ ]:
atracao = dispersao_com_r(
    dados_trabalho,
    "investimento_divulgacao",
    "entradas",
    "Investimento em divulgação × entradas",
    "Investimento em divulgação (R$)",
    "Entradas de pessoas"
)


### Registro 2 — evidência sobre Atração

**3. Qual é a direção da relação? Considerando o valor de `r` e o espalhamento dos pontos, o que essa análise permite afirmar sobre a associação entre investimento em divulgação e entradas?**

**Resposta:**


**4. Qual é o limite dessa evidência para responder à pergunta principal do projeto?**

**Resposta:**

## Etapa 3 — Investiguem a frente Conversão

Nesta frente, o indicador de aproveitamento é `cupons_por_100_entradas`.

Primeiro, examinem duas condições registradas diretamente no funcionamento da loja:

1. disponibilidade de produtos na abertura;
2. tempo médio até o primeiro atendimento.

Executem a célula abaixo.

In [ ]:
conversao_disponibilidade = dispersao_com_r(
    dados_trabalho,
    "disponibilidade_produtos_pct",
    "cupons_por_100_entradas",
    "Disponibilidade de produtos × aproveitamento",
    "Disponibilidade de produtos na abertura (%)",
    "Cupons por 100 entradas"
)

conversao_espera = dispersao_com_r(
    dados_trabalho,
    "espera_media_min",
    "cupons_por_100_entradas",
    "Espera média × aproveitamento",
    "Espera média (min)",
    "Cupons por 100 entradas"
)


### Registro 3 — disponibilidade e espera

**5. Para cada relação, registrem a direção observada e a evidência fornecida pelo gráfico e pelo valor de `r`.**

**Disponibilidade de produtos × aproveitamento:**  
**Resposta:**


**Espera média × aproveitamento:**  
**Resposta:**


**6. Qual das duas relações apresenta associação linear mais intensa? Como vocês identificaram isso?**

**Resposta:**

### Horas de equipe — primeira leitura

Agora, examinem a relação entre `horas_equipe` e `cupons_por_100_entradas` usando **todas as lojas juntas**.

Executem a célula e registrem a primeira interpretação antes de avançar.

In [ ]:
equipe_global = dispersao_com_r(
    dados_trabalho,
    "horas_equipe",
    "cupons_por_100_entradas",
    "Horas de equipe × aproveitamento — todas as lojas",
    "Horas de equipe",
    "Cupons por 100 entradas"
)


### Registro 4 — interpretação inicial

**7. Considerando todas as lojas juntas, registrem a direção e a intensidade aproximada da associação entre horas de equipe e aproveitamento. Sustentem a resposta no gráfico e no valor de r.**

**Resposta:**

### Do resultado global aos grupos

A análise anterior reuniu todas as lojas e produziu um resultado global para `horas_equipe × cupons_por_100_entradas`.

Agora façam uma pergunta adicional:

> **Esse comportamento global também descreve o que acontece dentro de cada loja?**

Comecem comparando as distribuições de `horas_equipe` e de `cupons_por_100_entradas` entre as lojas.

Os **boxplots** permitem comparar, entre grupos, a posição central, a dispersão e valores extremos. Eles não mostram, sozinhos, como duas variáveis se relacionam dentro de cada loja; servem aqui para verificar se as lojas apresentam perfis diferentes e se vale aprofundar a análise por grupos.

Executem a célula abaixo.


In [ ]:
ordem_lojas = sorted(
    dados_trabalho["loja"].dropna().unique(),
    key=lambda x: int(str(x).replace("L", ""))
)

# Boxplot 1 — horas de equipe por loja
grupos_horas = [
    dados_trabalho.loc[dados_trabalho["loja"] == loja, "horas_equipe"].dropna()
    for loja in ordem_lojas
]

plt.figure(figsize=(7, 4.5))
plt.boxplot(grupos_horas)
plt.xticks(range(1, len(ordem_lojas) + 1), ordem_lojas)
plt.xlabel("Loja")
plt.ylabel("Horas de equipe")
plt.title("Distribuição das horas de equipe por loja")
plt.grid(axis="y", alpha=0.15)
plt.tight_layout()
plt.show()

# Boxplot 2 — aproveitamento do movimento por loja
grupos_aproveitamento = [
    dados_trabalho.loc[
        dados_trabalho["loja"] == loja,
        "cupons_por_100_entradas"
    ].dropna()
    for loja in ordem_lojas
]

plt.figure(figsize=(7, 4.5))
plt.boxplot(grupos_aproveitamento)
plt.xticks(range(1, len(ordem_lojas) + 1), ordem_lojas)
plt.xlabel("Loja")
plt.ylabel("Cupons por 100 entradas")
plt.title("Distribuição do aproveitamento por loja")
plt.grid(axis="y", alpha=0.15)
plt.tight_layout()
plt.show()


### Registro 5 — diferenças entre as lojas

**8. Examinem os dois boxplots. Registrem duas diferenças observadas entre as lojas: uma referente a `horas_equipe` e outra a `cupons_por_100_entradas`.**

**Horas de equipe — diferença observada:**


**Aproveitamento — diferença observada:**


### Determinem a faixa usando o critério fornecido

As lojas recebem volumes de movimento diferentes. Para comparar classes formadas por **loja + faixa de entradas**, apliquem o seguinte critério: entre as três faixas, determinem aquela cujo **menor número de registros por loja seja o maior**. Usem a coluna “menor n entre lojas”. Esse critério evita utilizar uma faixa com muito poucos registros em alguma loja; não comprova representatividade estatística.

A caracterização mostrou que a mediana de `entradas` da rede está próxima de 180. Por isso, comparem três faixas centrais de 50 entradas: **100–149**, **150–199** e **200–249**.

A célula abaixo mostra quantos loja-dias de cada loja existem em cada faixa. O roteiro utiliza a faixa de 150–199 entradas nas comparações seguintes. Confiram essa escolha aplicando o critério fornecido às contagens da tabela, sem usar o resultado da correlação como critério.


In [ ]:
faixas_comparacao = [
    (100, 149),
    (150, 199),
    (200, 249),
]

linhas_faixas = []
for inicio, fim in faixas_comparacao:
    recorte_faixa = dados_trabalho[dados_trabalho["entradas"].between(inicio, fim)]
    contagens = recorte_faixa.groupby("loja").size().reindex(ordem_lojas, fill_value=0)
    linha = {"faixa de entradas": f"{inicio}–{fim}"}
    linha.update({loja: int(contagens.loc[loja]) for loja in ordem_lojas})
    linha["menor n entre lojas"] = int(contagens.min())
    linhas_faixas.append(linha)

cobertura_faixas = pd.DataFrame(linhas_faixas).set_index("faixa de entradas")
display(cobertura_faixas)

# A faixa 150–199 tem o maior número mínimo de registros por loja entre as três faixas comparadas.
faixa_inicio, faixa_fim = 150, 199
dados_faixa = dados_trabalho[dados_trabalho["entradas"].between(faixa_inicio, faixa_fim)].copy()

print(
    f"Faixa usada nas comparações seguintes: {faixa_inicio}–{faixa_fim} entradas | "
    f"{len(dados_faixa)} loja-dias"
)


### Registro 6 — por que usar 150–199 entradas?

**9. Entre as três faixas, determinem aquela cujo menor número de registros por loja seja o maior. Usem a coluna “menor n entre lojas” e justifiquem a escolha.**

**Resposta:**


### Examinem o resultado global e os grupos

Agora observem **horas de equipe × aproveitamento** em quatro leituras:

1. **Todas as lojas e todos os dias**;
2. **Cada loja, usando todos os seus dias**;
3. **Todas as lojas, somente com 150–199 entradas**;
4. **Cada loja, somente com 150–199 entradas**.

Nesta aula, o objetivo é **executar e examinar** essas quatro leituras. Registrem o que acontece com a **direção** e a **intensidade** da relação quando o conjunto é observado nesses diferentes níveis.

Depois, examinem os exemplos L4 × L5 e L8 × L3 na faixa de 150–199 entradas e registrem apenas os contrastes observados. A interpretação do significado desses resultados para a decisão da rede será realizada na Aula 4.


In [ ]:
# 1) Todas as lojas e todos os dias — visão global
pares_global = dados_trabalho[["horas_equipe", "cupons_por_100_entradas"]].dropna()
r_global = pares_global["horas_equipe"].corr(pares_global["cupons_por_100_entradas"])
print(
    f"1. Todas as lojas | todos os dias: n = {len(pares_global)} | "
    f"r = {r_global:+.3f}"
)

# 2) Cada loja e todos os dias — relação dentro de cada unidade
linhas_loja_todos_dias = []
for loja in ordem_lojas:
    recorte = dados_trabalho[dados_trabalho["loja"] == loja]
    pares = recorte[["horas_equipe", "cupons_por_100_entradas"]].dropna()
    linhas_loja_todos_dias.append({
        "loja": loja,
        "n_todos_dias": len(pares),
        "r_todos_dias": pares["horas_equipe"].corr(
            pares["cupons_por_100_entradas"]
        ),
    })

por_loja_todos_dias = pd.DataFrame(linhas_loja_todos_dias).set_index("loja")
print("\n2. Cada loja | todos os dias")
display(por_loja_todos_dias.round(3))

# 3) Todas as lojas, somente na faixa escolhida — aproximação do nível de movimento
pares_faixa_rede = dados_faixa[["horas_equipe", "cupons_por_100_entradas"]].dropna()
r_faixa_rede = pares_faixa_rede["horas_equipe"].corr(
    pares_faixa_rede["cupons_por_100_entradas"]
)
print(
    f"\n3. Todas as lojas | 150–199 entradas: n = {len(pares_faixa_rede)} | "
    f"r = {r_faixa_rede:+.3f}"
)

# 4) Cada loja, somente na faixa escolhida — mesma loja + movimento semelhante
resumo_classes = []
for loja in ordem_lojas:
    recorte = dados_faixa[dados_faixa["loja"] == loja].copy()
    pares = recorte[["horas_equipe", "cupons_por_100_entradas"]].dropna()
    resumo_classes.append({
        "loja": loja,
        "n_150_199": len(recorte),
        "mediana_horas_equipe": recorte["horas_equipe"].median(),
        "mediana_cupons_por_100": recorte["cupons_por_100_entradas"].median(),
        "r_150_199": pares["horas_equipe"].corr(
            pares["cupons_por_100_entradas"]
        ),
    })

resumo_classes = pd.DataFrame(resumo_classes).set_index("loja")
print("\n4. Cada loja | 150–199 entradas")
display(resumo_classes.round(3))

# Dois contrastes claros entre classes na mesma faixa de movimento.
print("\nExemplo 1 — L4 × L5")
display(resumo_classes.loc[["L4", "L5"]].round(3))

print("\nExemplo 2 — L8 × L3")
display(resumo_classes.loc[["L8", "L3"]].round(3))

# Visualizem a relação em um dos contrastes.
for loja in ["L4", "L5"]:
    dispersao_com_r(
        dados_faixa[dados_faixa["loja"] == loja],
        "horas_equipe",
        "cupons_por_100_entradas",
        f"Horas de equipe × aproveitamento — {loja}, 150–199 entradas",
        "Horas de equipe",
        "Cupons por 100 entradas"
    )


### Registro 7 — registrem os resultados observados

**10. Na faixa de 150–199 entradas, examinem L4 × L5 e L8 × L3. Em cada par, qual loja apresenta menor mediana de horas de equipe e maior mediana de aproveitamento?**

**Resposta:**


**11. Examinem as quatro leituras de `horas_equipe × cupons_por_100_entradas`. Registrem a direção e a intensidade aproximada da relação em cada nível.**

- **Todas as lojas e todos os dias:**


- **Cada loja, usando todos os dias:**


- **Todas as lojas, somente com 150–199 entradas:**


- **Cada loja, somente com 150–199 entradas:**


> Guardem esse registro. Na Aula 4, vocês vão explicar o que essas diferenças significam e como elas podem orientar uma análise mais aprofundada.


## Etapa 4 — Consolidem o registro da aula

Nesta aula, vocês executaram a parte do plano correspondente à **caracterização**, à **Atração** e à **Conversão**. A frente de **Resultado** permanece para a próxima aula.

### Registro 8 — evidências produzidas

**12. Registrem uma evidência numérica ou visual produzida na frente de Atração e duas evidências produzidas na frente de Conversão. Não é necessário escolher ainda qual delas é mais importante.**

**Atração — evidência:**


**Conversão — evidência 1:**


**Conversão — evidência 2:**


**13. Qual frente planejada na Aula 2 ainda precisa ser executada na próxima aula?**

**Resposta:**


Ao terminar, salvem o notebook preenchido. Na próxima aula, vocês vão interpretar e integrar essas evidências, aprofundar a frente de Resultado e formular a recomendação inicial do projeto.


## Critérios para avaliação da atividade

- Executa as células e examina estatísticas, distribuições, gráficos de dispersão e correlações de acordo com o plano da investigação.
- Registra evidências produzidas nas frentes de Atração e Conversão com apoio em resultados visuais e/ou numéricos.
- Interpreta corretamente valores ausentes de `espera_media_min`, sem tratá-los como zero.
- Examina, por boxplots, diferenças nas distribuições de horas de equipe e de aproveitamento entre as lojas.
- Aplica o critério fornecido para determinar a faixa com o maior número mínimo de registros por loja, sem usar a correlação para escolher o recorte.
- Examina a relação entre horas de equipe e aproveitamento em quatro níveis e registra como direção e intensidade mudam entre eles.
- Mantém os registros no nível de associação observado nos dados, sem atribuir causalidade.
- Identifica a frente de Resultado como etapa ainda pendente para a próxima aula.

**Entrega:** um notebook preenchido e salvo por dupla ao final da aula. Na próxima aula, os registros serão interpretados e integrados ao aprofundamento da frente de Resultado.


---

**Sobre os dados**

O conjunto `rede_papelarias.csv` foi elaborado para fins didáticos e simula registros de operação diária de uma pequena rede de lojas. As conclusões do projeto descrevem somente esse conjunto e não devem ser generalizadas para empresas reais.

**Nota metodológica**

As correlações são calculadas com os pares válidos de cada relação. Quando `espera_media_min` está ausente, o registro é excluído somente da análise que depende dessa variável; as demais informações do mesmo loja-dia continuam disponíveis para outras análises.